# X-OPT - EXACT FACILITY REMOVAL REOPTIMIZATION

This notebook studies a facility-removal reoptimization scenario over the first ten OR-Library p-median instances. For each instance, the workflow is:

1. solve the original instance with the exact `python-mip` model and with the metaheuristic;
2. extract the max-p-cut, highest k-core, densest subgraph, and the union between the highest k-core and the densest subgraph from the metaheuristic long-term-memory co-occurrence graph;
3. take the exact optimal facility set, remove one facility from it, and reoptimize the exact model under five schemes:
   - baseline: keep the remaining `p - 1` exact facilities fixed and choose one replacement while forbidding the removed node;
   - highest k-core: keep the remaining `p - 1` exact facilities fixed and choose the replacement only from the highest k-core candidates;
   - densest subgraph: keep the remaining `p - 1` exact facilities fixed and choose the replacement only from the densest-subgraph candidates;
   - highest k-core union densest subgraph: keep the remaining `p - 1` exact facilities fixed and choose the replacement only from the union of those two candidate sets;
   - max-p-cut: keep the remaining `p - 1` exact facilities fixed and choose the replacement only from the max-p-cut group that contains the removed facility, while forbidding the removed node.

By default the removed facility is the first one in the sorted exact optimal solution, but the position is configurable.

### SETUP

In [1]:
from __future__ import annotations

import gc
import os
import sys

import numpy  as np
import pandas as pd

from pathlib         import Path
from tqdm.auto       import tqdm
from IPython.display import display
from time            import perf_counter
from mip             import BINARY, Model, OptimizationStatus, minimize, xsum


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

In [2]:
from lib.paths     import find_project_root      , \
                          instance_sort_key      , \
                          canonical_instance_name

from lib.instances import list_orlibrary_instances, \
                          apply_instance_selection, \
                          read_instance_metadata  , \
                          load_best_known_costs_to_dict_id

from lib.graph     import read_orlibrary_graph    , \
                          all_pairs_shortest_paths
from lib.mip       import extract_open_facilities_candidates

from lib.explain   import extract_structure_insights
from lib.metrics   import compute_solution_cost   , \
                          gap_to_reference_percent

from lib.utils     import finite_or_none          , \
                          parse_optional_int_env  , \
                          parse_optional_float_env, \
                          as_sorted_tuple         , \
                          format_facilities       , \
                          format_groups

In [3]:
PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

PROJECT_ROOT = /home/rei-luisinho/xopt


In [4]:
import pymedian

### CONFIGURATION

In [5]:
INSTANCES_DIR = PROJECT_ROOT  / "instances"
PMEDOPT_PATH  = INSTANCES_DIR / "pmedopt.txt"
OUTPUT_DIR    = PROJECT_ROOT  / "notebooks" / "experiments_sbpo" / "artifacts"

RAW_RESULTS_CSV = OUTPUT_DIR / "facility_removal_reoptimization_raw.csv"
STRUCTURES_CSV  = OUTPUT_DIR / "facility_removal_reoptimization_structures.csv"
SUMMARY_CSV     = OUTPUT_DIR / "facility_removal_reoptimization_summary.csv"
FAILURES_CSV    = OUTPUT_DIR / "facility_removal_reoptimization_failures.csv"

INSTANCE_FILTER = os.getenv             ("FACREM_INSTANCE_FILTER")
MAX_INSTANCES   = parse_optional_int_env("FACREM_MAX_INSTANCES"  ) or 20

RESTARTS       = parse_optional_int_env  ("FACREM_META_RESTARTS"           ) or 8
MAX_ITER       = parse_optional_int_env  ("FACREM_META_MAX_ITER"           ) or 25
FACTOR         = parse_optional_int_env  ("FACREM_META_FACTOR"             ) or 1
TOP_FRACTION   = parse_optional_float_env("FACREM_TOP_FRACTION"            ) or 0.10
DETAILS_FORMAT = os.getenv               ("FACREM_DETAILS_FORMAT", "binary")

EXACT_TIME_LIMIT_SECONDS = parse_optional_int_env("FACREM_EXACT_TIME_LIMIT_SECONDS") or 400
REOPT_TIME_LIMIT_SECONDS = parse_optional_int_env("FACREM_REOPT_TIME_LIMIT_SECONDS") or 180
GLOBAL_SEED              = parse_optional_int_env("FACREM_GLOBAL_SEED"             ) or 42
MAX_P_CUT_RESTARTS       = parse_optional_int_env("FACREM_MAX_P_CUT_RESTARTS"      ) or 10
MAX_P_CUT_MAX_ITER       = parse_optional_int_env("FACREM_MAX_P_CUT_MAX_ITER"      ) or 1000

REMOVED_FACILITY_POSITION = parse_optional_int_env("FACREM_REMOVED_FACILITY_POSITION") or 0

SAVE_RESULTS_CSV = os.getenv("FACREM_SAVE_RESULTS_CSV", "1") != "0"

if DETAILS_FORMAT != "binary":
    raise ValueError(
        "This notebook expects FACREM_DETAILS_FORMAT='binary' so the interpretability pipeline can build the co-occurrence matrix."
    )

In [6]:
ALL_INSTANCE_NAMES = list_orlibrary_instances(INSTANCES_DIR)
INSTANCE_NAMES     = apply_instance_selection(
    ALL_INSTANCE_NAMES,
    pattern=INSTANCE_FILTER,
    limit  =MAX_INSTANCES  ,
)

BEST_KNOWN_COSTS = load_best_known_costs_to_dict_id(PMEDOPT_PATH)


if not INSTANCE_NAMES:
    raise FileNotFoundError(
        f"No instances were selected from {INSTANCES_DIR}."
    )

In [7]:
print(f"Instances discovered : {len(ALL_INSTANCE_NAMES)}")
print(f"Instances selected   : {len(INSTANCE_NAMES    )}")

print()

print("Instances to analyze:")
print(", ".join(Path(name).stem for name in INSTANCE_NAMES))

Instances discovered : 40
Instances selected   : 20

Instances to analyze:
pmed1, pmed2, pmed3, pmed4, pmed5, pmed6, pmed7, pmed8, pmed9, pmed10, pmed11, pmed12, pmed13, pmed14, pmed15, pmed16, pmed17, pmed18, pmed19, pmed20


In [8]:
print(f"Exact model time limit (s)      : {EXACT_TIME_LIMIT_SECONDS}")
print(f"Reoptimization time limit (s)   : {REOPT_TIME_LIMIT_SECONDS}")
print(f"Metaheuristic parameters        : restarts={          RESTARTS:2d}, max_iter={          MAX_ITER:4d}, factor={FACTOR   }")
print(f"Max-p-Cut parameters            : restarts={MAX_P_CUT_RESTARTS:2d}, max_iter={MAX_P_CUT_MAX_ITER:4d}, seed={GLOBAL_SEED}")

Exact model time limit (s)      : 400
Reoptimization time limit (s)   : 180
Metaheuristic parameters        : restarts= 8, max_iter=  25, factor=1
Max-p-Cut parameters            : restarts=10, max_iter=1000, seed=42


In [9]:
print(f"Top fraction kept from the LTM : {TOP_FRACTION         :.0%}")
print(f"Removed facility position      : {REMOVED_FACILITY_POSITION}")

Top fraction kept from the LTM : 10%
Removed facility position      : 0


In [10]:
print(f"Raw results CSV : {RAW_RESULTS_CSV}")
print(f"Structures CSV  : {STRUCTURES_CSV }")
print(f"Summary CSV     : {SUMMARY_CSV    }")

Raw results CSV : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_reoptimization_raw.csv
Structures CSV  : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_reoptimization_structures.csv
Summary CSV     : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_reoptimization_summary.csv


### HELPERS

In [11]:
def choose_removed_facility(
    open_facilities: tuple[int, ...] | list[int],
    position       : int                        ,
) -> int:
    facilities = as_sorted_tuple(open_facilities)

    if not facilities:
        raise ValueError(
            "No facilities are available to remove."
        )

    if position < 0 or position >= len(facilities):
        raise ValueError(
            f"FACREM_REMOVED_FACILITY_POSITION={position} is outside the optimal solution size {len(facilities)}."
        )

    return facilities[position]


def find_group_containing(
    groups   : list[tuple[int, ...]],
    facility : int                  ,
) -> tuple[int, ...]:
    for group in groups:
        if facility in group:
            return tuple(sorted(group))

    raise ValueError(
        f"Facility {facility} was not found in any Max-p-Cut group."
    )


def build_max_p_cut_local_replacement_pool(
    groups           : list[tuple[int, ...]],
    removed_facility : int                  ,
) -> tuple[tuple[int, ...], tuple[int, ...], str | None]:
    try:
        removed_group = find_group_containing(groups, removed_facility)
    except ValueError as exc:
        return tuple(), tuple(), f"{exc} Max-p-cut local reoptimization will be marked infeasible."

    replacement_pool = tuple(
        value
        for value in removed_group
        if  value != removed_facility
    )

    if not replacement_pool:
        return removed_group, replacement_pool, (
            "The Max-p-Cut group that contains the removed facility has no "
            "replacement candidate after forbidding the removed facility."
        )

    return removed_group, replacement_pool, None

### EXACT MODEL AND REOPTIMIZATION

In [12]:
def build_pmedian_model(
    distances : np.ndarray,
    p         : int       ,
    *,
    allowed_facilities    : tuple[int, ...] | list[int] | set[int] | None = None,
    fixed_open_facilities : tuple[int, ...] | list[int] | set[int] | None = None,
    forbidden_facilities  : tuple[int, ...] | list[int] | set[int] | None = None,
) -> tuple[Model, list[list], list, list[int]]:
    if distances.ndim != 2 or distances.shape[0] != distances.shape[1]:
        raise ValueError("Distances must be a square 2D array.")

    n = distances.shape[0]
    if not (1 <= p <= n):
        raise ValueError("P must satisfy 1 <= p <= n.")

    fixed_set     = set() if fixed_open_facilities is None else {int(value) for value in fixed_open_facilities}
    forbidden_set = set() if forbidden_facilities  is None else {int(value) for value in forbidden_facilities }

    if not fixed_set.isdisjoint(forbidden_set):
        raise ValueError("A fixed-open facility cannot also be forbidden.")

    if allowed_facilities is None:
        candidate_set = set(range(n))
    else:
        candidate_set = {int(value) for value in allowed_facilities}

    candidate_set.difference_update(forbidden_set)
    candidate_set.update           (fixed_set    )

    candidate_facilities = sorted(candidate_set)

    if len(candidate_facilities) < p:
        raise ValueError(
            f"Candidate facility pool is too small: {len(candidate_facilities)} < {p}."
        )

    if len(fixed_set) > p:
        raise ValueError(
            f"Too many fixed facilities: {len(fixed_set)} > {p}."
        )

    model = Model(solver_name="CBC")
    model.verbose = 0

    x: list[list] = []
    y             = [
        model.add_var(
            var_type=BINARY
        )
        for _ in candidate_facilities
    ]

    for _ in range(n):
        x_row = [
            model.add_var(
                var_type=BINARY
            )
            for _ in candidate_facilities
        ]
        x.append(x_row)

        model.add_constr(xsum(x_row) == 1)

        for pos in range(len(candidate_facilities)):
            model.add_constr(x_row[pos] <= y[pos])

    facility_to_pos = {
        facility: pos
        for pos, facility in enumerate(candidate_facilities)
    }

    for facility in fixed_set:
        if facility not in facility_to_pos:
            raise ValueError(
                f"Fixed facility {facility} is not available in the candidate pool."
            )

        model.add_constr(y[facility_to_pos[facility]] == 1)

    model.add_constr(xsum(y) == p)

    model.objective = minimize(
        xsum(
            float(distances[i, facility]) * x[i][pos]
            for i in range(n)
            for pos, facility in enumerate(candidate_facilities)
        )
    )

    return model, x, y, candidate_facilities


def optimize_model(
    model              : Model     ,
    time_limit_seconds : int | None,
) -> OptimizationStatus:
    if time_limit_seconds is None:
        return model.optimize()

    return model.optimize(
        max_seconds=float(time_limit_seconds)
    )


def describe_solution_change(
    reference_open: tuple[int, ...] | list[int],
    candidate_open: tuple[int, ...] | list[int],
) -> dict[str, object]:
    reference_set = set(as_sorted_tuple(reference_open))
    candidate_set = set(as_sorted_tuple(candidate_open))

    overlap = sorted(reference_set.intersection(candidate_set))
    dropped = sorted(reference_set - candidate_set)
    added   = sorted(candidate_set - reference_set)

    return {
        "overlap_with_original_optimum_count"    : len(overlap),
        "overlap_with_original_optimum_fraction" : len(overlap) / max(1, len(reference_set)),

        "dropped_optimal_facilities_count" : len(dropped),
        "new_facilities_count"             : len(added  ),
        "dropped_optimal_facilities"       : format_facilities(dropped),
        "new_facilities"                   : format_facilities(added  ),
    }


def solve_pmedian_variant(
    *,
    variant             : str,
    variant_order       : int,

    distances           : np.ndarray,
    p                   : int       ,
    time_limit_seconds  : int | None,

    allowed_facilities    : tuple[int, ...] | list[int] | set[int] | None = None,
    fixed_open_facilities : tuple[int, ...] | list[int] | set[int] | None = None,
    forbidden_facilities  : tuple[int, ...] | list[int] | set[int] | None = None,
) -> tuple[dict[str, object], tuple[int, ...]]:
    n = distances.shape[0]

    fixed_set     = set() if fixed_open_facilities is None else {int(value) for value in fixed_open_facilities}
    forbidden_set = set() if forbidden_facilities  is None else {int(value) for value in forbidden_facilities }

    if allowed_facilities is None:
        candidate_set = set(range(n))
    else:
        candidate_set = {int(value) for value in allowed_facilities}

    candidate_set.difference_update(forbidden_set)
    candidate_set.update           (fixed_set    )

    candidate_facilities = sorted(candidate_set)

    row = {
        "variant"       : variant      ,
        "variant_order" : variant_order,

        "candidate_facility_count" : len(candidate_facilities  ),
        "fixed_facility_count"     : len(fixed_set             ),
        "forbidden_facility_count" : len(forbidden_set         ),

        "candidate_facility_fraction" : len(candidate_facilities) / max(1, n),
        "fixed_facilities"            : format_facilities(fixed_set    ),
        "forbidden_facilities"        : format_facilities(forbidden_set),

        "objective_value"       : np.nan,
        "model_build_seconds"   : np.nan,
        "solve_seconds"         : np.nan,
        "total_variant_seconds" : np.nan,
        "open_facilities_count" : 0 ,
        "open_facilities"       : "",

        "status"        : "ERROR",
        "error_message" : None   ,
    }

    if len(candidate_facilities) < p:
        row.update(
            {
                "status"        : "INFEASIBLE_POOL",
                "error_message" : f"Candidate facility pool smaller than p: {len(candidate_facilities)} < {p}.",
            }
        )

        return row, tuple()

    if len(fixed_set) > p:
        row.update(
            {
                "status"        : "INVALID_FIXED_SET",
                "error_message" : f"Too many fixed facilities: {len(fixed_set)} > {p}.",
            }
        )

        return row, tuple()

    build_started_at = perf_counter()

    try:
        model, _, y, candidate_facilities = build_pmedian_model(
            distances,
            p        ,
            allowed_facilities   =candidate_facilities,
            fixed_open_facilities=fixed_set           ,
            forbidden_facilities =forbidden_set       ,
        )

        build_seconds = perf_counter() - build_started_at

        solve_started_at = perf_counter  ()
        status           = optimize_model(model, time_limit_seconds)
        solve_seconds    = perf_counter  () - solve_started_at

        has_incumbent = status in {
            OptimizationStatus.OPTIMAL ,
            OptimizationStatus.FEASIBLE,
        }

        solver_objective = finite_or_none(
            model.objective_value if has_incumbent else None
        )

        open_facilities     = tuple()
        validated_objective = None

        if has_incumbent:
            open_facilities = extract_open_facilities_candidates(
                candidate_facilities, y
            )

            if len(open_facilities) != p:
                raise ValueError(
                    f"Expected {p} open facilities, but recovered {len(open_facilities)}."
                )

            validated_objective = compute_solution_cost(distances, open_facilities)

            if solver_objective    is not None and \
               validated_objective is not None:
                if abs(solver_objective - validated_objective) > 1e-4:
                    raise ValueError(
                         "Solver objective and validated objective do not match: "
                        f"{solver_objective} vs {validated_objective}."
                    )

        objective_value = validated_objective if validated_objective is not None else solver_objective

        row.update(
            {
                "objective_value"       : objective_value if objective_value is not None else np.nan,
                "model_build_seconds"   : build_seconds   ,
                "solve_seconds"         : solve_seconds   ,
                "total_variant_seconds" : build_seconds + \
                                          solve_seconds   ,

                "open_facilities_count" : len              (open_facilities),
                "open_facilities"       : format_facilities(open_facilities),

                "status"        : getattr(status, "name", str(status)),
                "error_message" : None,
            }
        )

        return row, open_facilities
    except Exception as exc:
        row.update(
            {
                "status"              : "ERROR",
                "model_build_seconds" : perf_counter() - build_started_at,
                "error_message"       : f"{type(exc).__name__}: {exc}"   ,
            }
        )

        return row, tuple()
    finally:
        gc.collect()

### BENCHMARK EXECUTION

In [13]:
VARIANT_SPECS = [
    {
        "variant"      : "baseline",
        "variant_order": 1   ,
        "allowed_key"  : None,
    },
    {
        "variant"      : "max_p_cut",
        "variant_order": 2   ,
        "allowed_key"  : None,
    },
    {
        "variant"      : "highest_k_core",
        "variant_order": 3               ,
        "allowed_key"  : "highest_k_core_nodes",
    },
    {
        "variant"      : "densest_subgraph",
        "variant_order": 4              ,
        "allowed_key"  : "densest_nodes",
    },
    {
        "variant"      : "k_core_densest_union",
        "variant_order": 5                     ,
        "allowed_key"  : "k_core_densest_union_nodes",
    },
]

In [14]:
def build_pipeline_error_rows(
    instance_name  : str,
    instance_id    : str,

    metadata        : dict[str, int],
    best_known_cost : float | None  ,
    error_message   : str           ,
) -> list[dict[str, object]]:
    rows = []

    for spec in VARIANT_SPECS:
        rows.append(
            {
                "instance"       : instance_name,
                "instance_id"    : instance_id  ,
                "n"              : metadata.get("n", np.nan),
                "p"              : metadata.get("p", np.nan),
                "best_known_cost": best_known_cost if best_known_cost is not None else np.nan,

                "variant"        : spec["variant"      ],
                "variant_order"  : spec["variant_order"],
                "status"         : "PIPELINE_ERROR",
                "error_message"  : error_message   ,
            }
        )

    return rows


def build_pipeline_error_structure_row(
    instance_name  : str,
    instance_id    : str,

    metadata       : dict[str, int],
    best_known_cost: float | None  ,
    error_message  : str           ,
) -> dict[str, object]:
    return {
        "instance"       : instance_name,
        "instance_id"    : instance_id  ,
        "n"              : metadata.get("n", np.nan),
        "p"              : metadata.get("p", np.nan),
        "best_known_cost": best_known_cost if best_known_cost is not None else np.nan,

        "error_message"  : error_message,
    }


def run_single_instance_analysis(
    instance_name: str,
    *,
    restarts          : int  ,
    max_iter          : int  ,
    factor            : int  ,
    top_fraction      : float,
    details_format    : str  ,

    max_p_cut_restarts: int,
    max_p_cut_max_iter: int,
    global_seed       : int,

    exact_time_limit_seconds : int | None,
    reopt_time_limit_seconds : int | None,
    removed_facility_position: int,
) -> tuple[list[dict[str, object]], dict[str, object]]:
    instance_name = canonical_instance_name(instance_name)
    instance_path = INSTANCES_DIR / instance_name
    instance_id   = Path(instance_name).stem

    metadata        = read_instance_metadata(instance_path)
    best_known_cost = BEST_KNOWN_COSTS.get  (instance_id  )

    try:
        graph     = read_orlibrary_graph    (instance_path     )
        distances = all_pairs_shortest_paths(graph["adjacency"])
        p         = int(graph["p"])

        exact_row, exact_open = solve_pmedian_variant(
            variant            ="exact_reference",
            variant_order      =0                ,
            distances          =distances,
            p                  =p        ,
            time_limit_seconds =exact_time_limit_seconds,
        )

        if exact_row["status"] != "OPTIMAL":
            raise ValueError(
                 "The original exact model must be solved to OPTIMAL for this analysis. "
                f"Received status={exact_row['status']}."
            )

        meta_started_at  = perf_counter()
        summary, details = pymedian.solve_pmedian(
            instance_path,
            restarts      =restarts,
            max_iter      =max_iter,
            factor        =factor  ,
            details_format=details_format,
        )
        metaheuristic_runtime_seconds = perf_counter() - meta_started_at

        structure_started_at = perf_counter()
        insights             = extract_structure_insights(
            summary,
            details,
            top_fraction       =top_fraction      ,
            max_p_cut_restarts =max_p_cut_restarts,
            max_p_cut_max_iter =max_p_cut_max_iter,
            global_seed        =global_seed       ,
        )
        structure_runtime_seconds = perf_counter() - structure_started_at

        removed_facility = choose_removed_facility(
            exact_open, removed_facility_position
        )
        remaining_optimal = tuple(
            facility
            for facility in exact_open
            if  facility != removed_facility
        )

        removed_group, replacement_pool, max_p_cut_group_error = build_max_p_cut_local_replacement_pool(
            insights["max_p_cut_groups"], removed_facility
        )

        exact_objective = exact_row["objective_value"] if pd.notna(exact_row["objective_value"]) else None
        meta_best_set   = set(insights["best_facilities"])

        common = {
            "instance"       : instance_name  ,
            "instance_id"    : instance_id    ,
            "n"              : int(graph["n"]),
            "p"              : p              ,
            "best_known_cost": best_known_cost if best_known_cost is not None else np.nan,

            "exact_original_status"                    : exact_row["status"               ],
            "exact_original_objective"                 : exact_row["objective_value"      ],
            "exact_original_model_build_seconds"       : exact_row["model_build_seconds"  ],
            "exact_original_solve_seconds"             : exact_row["solve_seconds"        ],
            "exact_original_total_seconds"             : exact_row["total_variant_seconds"],
            "exact_original_open_facilities"           : format_facilities       (exact_open                      ),
            "exact_original_gap_to_best_known_percent" : gap_to_reference_percent(exact_objective, best_known_cost),

            "metaheuristic_best_cost"                : insights["best_cost"]        ,
            "metaheuristic_runtime_seconds"          : metaheuristic_runtime_seconds,
            "metaheuristic_gap_to_best_known_percent": gap_to_reference_percent(insights["best_cost"], best_known_cost),
            "metaheuristic_gap_to_exact_percent"     : gap_to_reference_percent(insights["best_cost"], exact_objective),
            "metaheuristic_best_facilities"          : format_facilities(insights["best_facilities" ]             ),
            "metaheuristic_exact_overlap_count"      : len(set(exact_open).intersection(meta_best_set)            ),
            "metaheuristic_exact_overlap_fraction"   : len(set(exact_open).intersection(meta_best_set)) / max(1, p),

            "structure_extraction_runtime_seconds" : structure_runtime_seconds,
            "pre_reoptimization_seconds"           : (
                exact_row["total_variant_seconds"] +
                metaheuristic_runtime_seconds      +
                structure_runtime_seconds
            ),

            "memory_size"               : insights["memory_size"              ],
            "top_solution_count"        : insights["top_solution_count"       ],
            "top_cost_cutoff"           : insights["top_cost_cutoff"          ],
            "cooccurrence_edges"        : insights["cooccurrence_edges"       ],
            "cooccurrence_total_weight" : insights["cooccurrence_total_weight"],

            "max_p_cut_fraction_cut"             : insights["max_p_cut_fraction_cut"             ],
            "max_p_cut_best_facility_separation" : insights["max_p_cut_best_facility_separation" ],
            "k_core_max_level"                   : insights["k_core_max_level"                   ],
            "k_core_candidate_count"             : insights["k_core_candidate_count"             ],
            "k_core_candidate_fraction"          : insights["k_core_candidate_fraction"          ],
            "k_core_best_set_recall"             : insights["k_core_best_set_recall"             ],
            "densest_subgraph_candidate_count"    : insights["densest_subgraph_candidate_count"    ],
            "densest_subgraph_candidate_fraction" : insights["densest_subgraph_candidate_fraction" ],
            "densest_subgraph_density"            : insights["densest_subgraph_density"            ],
            "densest_subgraph_best_set_recall"    : insights["densest_subgraph_best_set_recall"    ],
            "k_core_densest_union_candidate_count" : insights["k_core_densest_union_candidate_count" ],
            "k_core_densest_union_candidate_fraction": insights["k_core_densest_union_candidate_fraction"],
            "k_core_densest_union_best_set_recall" : insights["k_core_densest_union_best_set_recall" ],

            "removed_facility_zero_based"          : removed_facility         ,
            "removed_facility_position"            : removed_facility_position,
            "remaining_optimal_facilities"         : format_facilities(remaining_optimal),
            "remaining_optimal_facilities_count"   : len              (remaining_optimal),

            "max_p_cut_removed_group_found"        : int(bool(removed_group)),
            "max_p_cut_removed_group_size"         : len(removed_group      ),
            "max_p_cut_replacement_pool_size"      : len(replacement_pool   ),
        }

        structure_row = {
            "instance"       : instance_name  ,
            "instance_id"    : instance_id    ,
            "n"              : int(graph["n"]),
            "p"              : p              ,
            "best_known_cost": best_known_cost if best_known_cost is not None else np.nan,

            "exact_original_objective"       : exact_row["objective_value"],
            "metaheuristic_best_cost"        : insights ["best_cost"      ],
            "exact_original_open_facilities" : format_facilities(exact_open                 ),
            "metaheuristic_best_facilities"  : format_facilities(insights["best_facilities"]),

            "removed_facility"              : removed_facility + 1     ,
            "removed_facility_position"     : removed_facility_position,
            "remaining_optimal_facilities"  : format_facilities(remaining_optimal  ),
            "removed_facility_in_meta_best" : int(removed_facility in meta_best_set),

            "max_p_cut_group_error"      : max_p_cut_group_error,
            "max_p_cut_removed_group"    : format_facilities(removed_group               ),
            "max_p_cut_replacement_pool" : format_facilities(replacement_pool            ),
            "max_p_cut_groups"           : format_groups    (insights["max_p_cut_groups"]),

            "highest_k_core_nodes"      : format_facilities(insights["highest_k_core_nodes"     ]),
            "densest_nodes"             : format_facilities(insights["densest_nodes"            ]),
            "k_core_densest_union_nodes": format_facilities(insights["k_core_densest_union_nodes"]),

            "error_message" : None,
        }

        rows: list[dict[str, object]] = []

        for spec in VARIANT_SPECS:
            allowed_facilities    = None
            fixed_open_facilities = remaining_optimal

            if spec["variant"] == "max_p_cut":
                allowed_facilities = replacement_pool
            elif spec["allowed_key"] is not None:
                allowed_facilities = tuple(sorted(insights[spec["allowed_key"]]))

            row, open_facilities = solve_pmedian_variant(
                variant             =spec["variant"      ],
                variant_order       =spec["variant_order"],
                distances           =distances,
                p                   =p        ,
                time_limit_seconds  =reopt_time_limit_seconds,

                allowed_facilities   =allowed_facilities   ,
                fixed_open_facilities=fixed_open_facilities,
                forbidden_facilities =(removed_facility,)  ,
            )

            if spec["variant"] == "max_p_cut" and max_p_cut_group_error is not None:
                row["error_message"] = max_p_cut_group_error

            row.update(common)
            row.update(describe_solution_change(exact_open, open_facilities))

            row["degradation_vs_exact_percent"] = gap_to_reference_percent(
                row["objective_value"] if pd.notna(row["objective_value"]) else None,
                exact_objective                                                     ,
            )
            row["total_pipeline_seconds"      ] = common["pre_reoptimization_seconds"] + (
                row["total_variant_seconds"] if pd.notna(row["total_variant_seconds"]) else 0.0
            )

            rows.append(row)

        return rows, structure_row
    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"

        return (
            build_pipeline_error_rows(
                instance_name  ,
                instance_id    ,
                metadata       ,
                best_known_cost,
                error_message  ,
            ),
            build_pipeline_error_structure_row(
                instance_name  ,
                instance_id    ,
                metadata       ,
                best_known_cost,
                error_message  ,
            ),
        )


def sort_by_instance(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty or "instance" not in df.columns:
        return df

    ordered = df.copy()

    ordered["instance_order"] = ordered["instance"].map(
        lambda value: instance_sort_key(value)[0]
    )

    return (
        ordered.sort_values(["instance_order", "instance"])
               .drop       (columns="instance_order"      )
               .reset_index(drop   =True)
    )


def run_benchmark(
    instance_names: list[str],
    *,
    restarts          : int  ,
    max_iter          : int  ,
    factor            : int  ,
    top_fraction      : float,
    details_format    : str  ,

    max_p_cut_restarts: int,
    max_p_cut_max_iter: int,
    global_seed       : int,

    exact_time_limit_seconds : int | None,
    reopt_time_limit_seconds : int | None,
    removed_facility_position: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not instance_names:
        raise ValueError("The benchmark requires at least one instance.")

    rows           : list[dict[str, object]] = []
    structure_rows : list[dict[str, object]] = []

    for instance_name in tqdm(
        instance_names,
        total        =len(instance_names         ),
        desc         ="Facility-removal benchmark",
        dynamic_ncols=True                        ,
    ):
        instance_rows, structure_row = run_single_instance_analysis(
            instance_name,
            restarts          =restarts      ,
            max_iter          =max_iter      ,
            factor            =factor        ,
            top_fraction      =top_fraction  ,
            details_format    =details_format,

            max_p_cut_restarts=max_p_cut_restarts,
            max_p_cut_max_iter=max_p_cut_max_iter,
            global_seed       =global_seed       ,

            exact_time_limit_seconds =exact_time_limit_seconds ,
            reopt_time_limit_seconds =reopt_time_limit_seconds ,
            removed_facility_position=removed_facility_position,
        )

        rows          .extend(instance_rows)
        structure_rows.append(structure_row)

    results_df    = pd.DataFrame(rows)
    structures_df = pd.DataFrame(structure_rows)

    if not results_df.empty:
        results_df["variant_order"] = pd.to_numeric(
            results_df["variant_order"], errors="coerce"
        )

        results_df = sort_by_instance(results_df)
        results_df = results_df.sort_values(
            ["instance", "variant_order"],
            kind="stable",
        ).reset_index(drop=True)

    structures_df = sort_by_instance(structures_df)

    return results_df, structures_df


def add_forbidden_baseline_reference(
    results_df: pd.DataFrame
) -> pd.DataFrame:
    merged = results_df.copy()

    if merged.empty:
        return merged

    if "objective_value" not in merged.columns:
        merged["objective_value"] = np.nan
    if "status"          not in merged.columns:
        merged["status"] = np.nan

    baseline_df = (
        merged
        .loc[
            merged["variant"] == "baseline",
            [
                "instance",
                "status"  ,
                "objective_value"
            ],
        ]
        .rename(
            columns={
                "status"          : "baseline_status"         ,
                "objective_value" : "baseline_objective_value",
            }
        )
    )

    merged = merged.merge(baseline_df, on="instance", how="left")

    merged["gap_to_forbidden_baseline_percent"] = np.where(
        merged["objective_value"].notna()  & merged["baseline_objective_value"].notna(),
        100.0 * (merged["objective_value"] - merged["baseline_objective_value"]) / merged["baseline_objective_value"],
        np.nan,
    )

    merged["baseline_is_optimal"] = merged["baseline_status"].eq("OPTIMAL")

    return merged


def build_wide_summary(results_df: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return results_df.copy()

    instance_columns = [
        "instance"   ,
        "instance_id",
        "n"          ,
        "p"          ,
        "best_known_cost",

        "exact_original_status"                   ,
        "exact_original_objective"                ,
        "exact_original_gap_to_best_known_percent",
        "exact_original_total_seconds"            ,
        "exact_original_open_facilities"          ,

        "metaheuristic_best_cost"                ,
        "metaheuristic_gap_to_best_known_percent",
        "metaheuristic_gap_to_exact_percent"     ,
        "metaheuristic_runtime_seconds"          ,
        "metaheuristic_best_facilities"          ,
        "metaheuristic_exact_overlap_fraction"   ,

        "structure_extraction_runtime_seconds",
        "pre_reoptimization_seconds"          ,
        "memory_size"                         ,
        "top_solution_count"                  ,
        "top_cost_cutoff"                     ,
        "cooccurrence_edges"                  ,
        "cooccurrence_total_weight"           ,

        "max_p_cut_fraction_cut"             ,
        "max_p_cut_best_facility_separation" ,
        "k_core_max_level"                    ,
        "k_core_candidate_count"              ,
        "k_core_candidate_fraction"           ,
        "densest_subgraph_candidate_count"    ,
        "densest_subgraph_candidate_fraction" ,
        "densest_subgraph_density"            ,
        "k_core_densest_union_candidate_count",
        "k_core_densest_union_candidate_fraction",

        "removed_facility"               ,
        "removed_facility_position"      ,
        "remaining_optimal_facilities"   ,
        "max_p_cut_removed_group_found"  ,
        "max_p_cut_removed_group_size"   ,
        "max_p_cut_replacement_pool_size",
    ]

    metric_columns = [
        "status"                  ,
        "candidate_facility_count",
        "fixed_facility_count"    ,

        "objective_value"                  ,
        "degradation_vs_exact_percent"     ,
        "gap_to_forbidden_baseline_percent",

        "overlap_with_original_optimum_fraction",
        "new_facilities_count"                  ,
        "model_build_seconds"   ,
        "solve_seconds"         ,
        "total_variant_seconds" ,
        "total_pipeline_seconds",
    ]

    working = results_df.copy()
    for column in [*instance_columns, *metric_columns]:
        if column not in working.columns:
            working[column] = np.nan

    base_df = working[instance_columns].drop_duplicates("instance")

    wide_df = working.pivot(
        index  ="instance",
        columns="variant" ,
        values =metric_columns,
    )

    wide_df.columns = [f"{variant}__{metric}" for metric, variant in wide_df.columns]
    wide_df         = wide_df.reset_index()

    return base_df.merge(wide_df, on="instance", how="left")


def build_variant_aggregate(
    results_df: pd.DataFrame
) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    working = results_df.copy()

    for column in [
        "objective_value"                  ,
        "candidate_facility_fraction"      ,
        "total_variant_seconds"            ,
        "total_pipeline_seconds"           ,
        "degradation_vs_exact_percent"     ,
        "gap_to_forbidden_baseline_percent",
    ]:
        if column not in working.columns:
            working[column] = np.nan

    if "status" not in working.columns:
        working["status"] = np.nan

    return (
        working.groupby("variant", dropna=False)
               .agg    (
                   instances                    =("instance", "nunique"),

                   solved_instances             =("objective_value", lambda s: int( s.notna()      .sum())),
                   optimal_instances            =("status"         , lambda s: int((s == "OPTIMAL").sum())),

                   mean_candidate_fraction      =("candidate_facility_fraction"      , "mean"),
                   mean_total_variant_seconds   =("total_variant_seconds"            , "mean"),
                   mean_total_pipeline_seconds  =("total_pipeline_seconds"           , "mean"),
                   mean_degradation_vs_exact_pct=("degradation_vs_exact_percent"     , "mean"),
                   mean_gap_to_baseline_pct     =("gap_to_forbidden_baseline_percent", "mean"),
               )
               .reset_index()
               .sort_values("variant")
               .reset_index(drop=True)
    )

### APPLY

In [15]:
raw_results_df, structures_df = run_benchmark(
    INSTANCE_NAMES,
    restarts          =RESTARTS      ,
    max_iter          =MAX_ITER      ,
    factor            =FACTOR        ,
    top_fraction      =TOP_FRACTION  ,
    details_format    =DETAILS_FORMAT,

    max_p_cut_restarts=MAX_P_CUT_RESTARTS,
    max_p_cut_max_iter=MAX_P_CUT_MAX_ITER,
    global_seed       =GLOBAL_SEED       ,

    exact_time_limit_seconds =EXACT_TIME_LIMIT_SECONDS ,
    reopt_time_limit_seconds =REOPT_TIME_LIMIT_SECONDS ,
    removed_facility_position=REMOVED_FACILITY_POSITION,
)

Facility-removal benchmark:   0%|          | 0/20 [00:00<?, ?it/s]

In [16]:
results_df           = add_forbidden_baseline_reference(raw_results_df)
wide_summary_df      = build_wide_summary              (results_df)
variant_aggregate_df = build_variant_aggregate         (results_df)

failures_df = results_df[results_df["error_message"].notna()].copy()

In [17]:
display(structures_df.head())

,instance,instance_id,n,p,best_known_cost,exact_original_objective,metaheuristic_best_cost,exact_original_open_facilities,metaheuristic_best_facilities,removed_facility,removed_facility_position,remaining_optimal_facilities,removed_facility_in_meta_best,max_p_cut_group_error,max_p_cut_removed_group,max_p_cut_replacement_pool,max_p_cut_groups,highest_k_core_nodes,densest_nodes,k_core_densest_union_nodes,error_message
0,pmed1.txt,pmed1,100,5,5819.0,5819.0,5819.0,6 12 64 90 98,6 12 64 90 98,7.0,0.0,12 64 90 98,1.0,NaN,1 5 6 7 9 17 18 19 26 31 33 40 42 46 55 67 76 78 82 91 92 94 99,1 5 7 9 17 18 19 26 31 33 40 42 46 55 67 76 78 82 91 92 94 99,2 3 8 10 12 13 14 16 32 41 45 48 50 51 53 54 58 74 89 93 | 4 22 23 24 25 36 37 38 43 77 84 87 88 96 97 98 | 1 5 6 7 ...,6 12 24 41 63 64 90 98,6 12 24 41 64 90 98,6 12 24 41 63 64 90 98,None
1,pmed2.txt,pmed2,100,10,4093.0,4093.0,4093.0,5 7 11 36 40 44 57 66 94 98,5 7 11 36 40 44 57 66 94 98,6.0,0.0,7 11 36 40 44 57 66 94 98,1.0,NaN,5 10 16 20 24 26 28 78,10 16 20 24 26 28 78,0 9 32 35 44 55 68 75 76 86 | 17 22 39 40 51 54 71 73 82 | 6 7 8 38 77 95 96 | 21 33 36 43 52 62 65 67 87 | 3 4 11 1...,5 7 11 26 36 39 40 44 57 66 90 94 98,5 7 11 36 40 44 57 66 90 94 98,5 7 11 26 36 39 40 44 57 66 90 94 98,None
2,pmed3.txt,pmed3,100,10,4250.0,4250.0,4250.0,4 8 12 20 25 35 47 54 68 98,4 8 12 20 25 35 47 54 68 98,5.0,0.0,8 12 20 25 35 47 54 68 98,1.0,NaN,4 17 27 41 50 58 69 73 86,17 27 41 50 58 69 73 86,8 18 28 29 33 37 46 52 53 55 65 77 88 94 96 | 15 30 45 54 70 80 81 91 | 3 21 38 39 48 61 62 64 67 68 79 92 97 | 6 9 ...,4 8 12 13 20 25 35 47 54 68 73 95 98,8 12 13 20 25 35 47 54 68 98,4 8 12 13 20 25 35 47 54 68 73 95 98,None
3,pmed4.txt,pmed4,100,20,3034.0,3034.0,3034.0,0 4 7 8 12 21 25 33 37 49 54 59 65 71 76 82 86 90 92 95,0 4 6 8 12 21 25 33 37 49 54 59 65 71 76 82 86 90 92 95,1.0,0.0,4 7 8 12 21 25 33 37 49 54 59 65 71 76 82 86 90 92 95,1.0,NaN,0 32 35 44 68 86,32 35 44 68 86,99 | 0 32 35 44 68 86 | 6 7 17 73 | 22 51 57 82 | 8 9 31 77 96 | 24 38 95 | 56 76 87 | 21 43 52 67 93 | 3 26 40 46 5...,4 5 6 7 8 9 12 21 25 33 37 49 50 54 59 65 71 76 82 86 90 92 95 99,5 7 8 9 12 21 25 33 37 49 50 54 59 65 71 76 82 86 90 92 95 99,4 5 6 7 8 9 12 21 25 33 37 49 50 54 59 65 71 76 82 86 90 92 95 99,None
4,pmed5.txt,pmed5,100,33,1355.0,1355.0,1355.0,0 3 6 8 13 18 24 25 27 29 32 35 36 37 40 48 50 52 54 57 65 68 69 72 74 80 81 83 84 87 93 94 96,0 3 6 8 13 18 24 25 27 29 32 36 37 40 48 50 52 53 54 57 64 68 69 72 74 80 81 83 84 87 93 94 96,1.0,0.0,3 6 8 13 18 24 25 27 29 32 35 36 37 40 48 50 52 54 57 65 68 69 72 74 80 81 83 84 87 93 94 96,1.0,NaN,0 7 44,7 44,6 74 | 35 53 | 0 7 44 | 32 33 86 | 17 31 52 73 | 24 82 | 22 48 51 56 71 | 96 | 40 | 38 57 95 | 62 68 76 | 37 | 21 25...,3 7 8 13 18 24 25 28 30 32 35 36 37 40 48 50 52 53 54 55 57 64 65 68 69 72 74 80 81 83 84 87 90 93 94 96 99,3 7 8 13 18 24 25 28 30 32 35 36 37 40 48 50 52 55 57 64 65 68 69 72 74 80 81 84 87 90 93 94 96 99,3 7 8 13 18 24 25 28 30 32 35 36 37 40 48 50 52 53 54 55 57 64 65 68 69 72 74 80 81 83 84 87 90 93 94 96 99,None


In [18]:
display(wide_summary_df.head().round(4))

,instance,instance_id,n,p,best_known_cost,exact_original_status,exact_original_objective,exact_original_gap_to_best_known_percent,exact_original_total_seconds,exact_original_open_facilities,metaheuristic_best_cost,metaheuristic_gap_to_best_known_percent,metaheuristic_gap_to_exact_percent,metaheuristic_runtime_seconds,metaheuristic_best_facilities,metaheuristic_exact_overlap_fraction,structure_extraction_runtime_seconds,pre_reoptimization_seconds,memory_size,top_solution_count,top_cost_cutoff,cooccurrence_edges,cooccurrence_total_weight,max_p_cut_fraction_cut,max_p_cut_best_facility_separation,k_core_max_level,k_core_candidate_count,k_core_candidate_fraction,densest_subgraph_candidate_count,densest_subgraph_candidate_fraction,densest_subgraph_density,k_core_densest_union_candidate_count,k_core_densest_union_candidate_fraction,removed_facility,removed_facility_position,remaining_optimal_facilities,max_p_cut_removed_group_found,max_p_cut_removed_group_size,max_p_cut_replacement_pool_size,baseline__status,densest_subgraph__status,highest_k_core__status,k_core_densest_union__status,max_p_cut__status,baseline__candidate_facility_count,densest_subgraph__candidate_facility_count,highest_k_core__candidate_facility_count,k_core_densest_union__candidate_facility_count,max_p_cut__candidate_facility_count,baseline__fixed_facility_count,densest_subgraph__fixed_facility_count,highest_k_core__fixed_facility_count,k_core_densest_union__fixed_facility_count,max_p_cut__fixed_facility_count,baseline__objective_value,densest_subgraph__objective_value,highest_k_core__objective_value,k_core_densest_union__objective_value,max_p_cut__objective_value,baseline__degradation_vs_exact_percent,densest_subgraph__degradation_vs_exact_percent,highest_k_core__degradation_vs_exact_percent,k_core_densest_union__degradation_vs_exact_percent,max_p_cut__degradation_vs_exact_percent,baseline__gap_to_forbidden_baseline_percent,densest_subgraph__gap_to_forbidden_baseline_percent,highest_k_core__gap_to_forbidden_baseline_percent,k_core_densest_union__gap_to_forbidden_baseline_percent,max_p_cut__gap_to_forbidden_baseline_percent,baseline__overlap_with_original_optimum_fraction,densest_subgraph__overlap_with_original_optimum_fraction,highest_k_core__overlap_with_original_optimum_fraction,k_core_densest_union__overlap_with_original_optimum_fraction,max_p_cut__overlap_with_original_optimum_fraction,baseline__new_facilities_count,densest_subgraph__new_facilities_count,highest_k_core__new_facilities_count,k_core_densest_union__new_facilities_count,max_p_cut__new_facilities_count,baseline__model_build_seconds,densest_subgraph__model_build_seconds,highest_k_core__model_build_seconds,k_core_densest_union__model_build_seconds,max_p_cut__model_build_seconds,baseline__solve_seconds,densest_subgraph__solve_seconds,highest_k_core__solve_seconds,k_core_densest_union__solve_seconds,max_p_cut__solve_seconds,baseline__total_variant_seconds,densest_subgraph__total_variant_seconds,highest_k_core__total_variant_seconds,k_core_densest_union__total_variant_seconds,max_p_cut__total_variant_seconds,baseline__total_pipeline_seconds,densest_subgraph__total_pipeline_seconds,highest_k_core__total_pipeline_seconds,k_core_densest_union__total_pipeline_seconds,max_p_cut__total_pipeline_seconds
0,pmed1.txt,pmed1,100,5,5819.0,OPTIMAL,5819.0,0.0,3.3284,6 12 64 90 98,5819.0,0.0,0.0,0.0537,6 12 64 90 98,1.0000,0.0259,3.4079,129.0,13.0,5856.0,40.0,130.0,1.0,1.0000,6.0,8.0,0.0800,7.0,0.0700,13.4286,8.0,0.0800,NaN,0.0,12 64 90 98,1.0,23.0,22.0,OPTIMAL,OPTIMAL,OPTIMAL,OPTIMAL,OPTIMAL,99.0,6.0,7.0,7.0,26.0,4.0,4.0,4.0,4.0,4.0,5862.0,6693.0,6693.0,6693.0,5979.0,0.738959,15.019763,15.019763,15.019763,2.749613,0.0,14.176049,14.176049,14.176049,1.995906,0.8,0.8,0.8,0.8,0.8,1.0,1.0,1.0,1.0,1.0,0.144222,0.007891,0.011396,0.009461,0.035909,0.875113,0.034604,0.047136,0.05372,0.339382,1.019335,0.042495,0.058532,0.06318,0.375291,4.427256,3.450415,3.466452,3.471101,3.783211
1,pmed10.txt,pmed10,200,67,1255.0,OPTIMAL,1255.0

In [19]:
display(variant_aggregate_df.round(4))

,variant,instances,solved_instances,optimal_instances,mean_candidate_fraction,mean_total_variant_seconds,mean_total_pipeline_seconds,mean_degradation_vs_exact_pct,mean_gap_to_baseline_pct
0,baseline,20,19,19,0.9946,5.2780,50.0290,0.2663,0.0000
1,densest_subgraph,20,19,19,0.1701,0.4328,45.1838,2.9106,2.6335
2,highest_k_core,20,19,19,0.1830,0.6395,45.3905,1.6169,1.3426
3,k_core_densest_union,20,19,19,0.1832,0.6478,45.3987,1.6169,1.3426
4,max_p_cut,20,19,19,0.2148,1.1087,45.8596,1.0523,0.7833


In [20]:
if SAVE_RESULTS_CSV:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    results_df     .to_csv(RAW_RESULTS_CSV, index=False)
    structures_df  .to_csv(STRUCTURES_CSV , index=False)
    wide_summary_df.to_csv(SUMMARY_CSV    , index=False)

    print(f"Raw results saved to : {RAW_RESULTS_CSV}")
    print(f"Structures saved to  : {STRUCTURES_CSV }")
    print(f"Summary saved to     : {SUMMARY_CSV    }")

    if not failures_df.empty:
        failures_df.to_csv(FAILURES_CSV, index=False)

        print(f"Failures saved to : {FAILURES_CSV}")

Raw results saved to : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_reoptimization_raw.csv
Structures saved to  : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_reoptimization_structures.csv
Summary saved to     : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_reoptimization_summary.csv
Failures saved to : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_reoptimization_failures.csv
